# 🎨 ComfyUI trên Google Colab (Free)

**Hướng dẫn:**
1. Vào menu `Runtime` → `Change runtime type` → chọn **T4 GPU** → Save
2. Chạy lần lượt **Cell 1 → Cell 2 → Cell 3** (bấm nút ▶ bên trái mỗi cell)
3. Ở Cell 3, đợi dòng link `https://xxxx.trycloudflare.com` hiện ra rồi bấm vào đó để mở giao diện ComfyUI

⚠️ Colab free sẽ mất toàn bộ file khi ngắt phiên — lần sau vào chạy lại từ Cell 1.

In [ ]:
# ===== CELL 1: Kiểm tra GPU + Cài ComfyUI =====
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os
os.chdir('/content')
!rm -rf /content/ComfyUI

# Clone ComfyUI (repo đang hoạt động bình thường, không phụ thuộc repo Stability-AI đã bị xóa)
!git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

os.chdir('/content/ComfyUI')
!pip install -q -r requirements.txt

print('\n✅ Cài ComfyUI xong! Chạy tiếp Cell 2.')

In [ ]:
# ===== CELL 2: Tải Model =====
# SD 1.5 bản fp16 (2.1GB) - mirror chính thức của Comfy-Org
# (link runwayml/stable-diffusion-v1-5 cũ đã bị gỡ khỏi Hugging Face)
!wget -c -O /content/ComfyUI/models/checkpoints/v1-5-pruned-emaonly-fp16.safetensors \
  "https://huggingface.co/Comfy-Org/stable-diffusion-v1-5-archive/resolve/main/v1-5-pruned-emaonly-fp16.safetensors"

# (TÙY CHỌN) Bỏ dấu # ở 2 dòng dưới nếu muốn thêm model ảnh người thật đẹp hơn:
#!wget -c -O /content/ComfyUI/models/checkpoints/RealisticVision51.safetensors \
#  "https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE/resolve/main/Realistic_Vision_V5.1_fp16-no-ema.safetensors"

!ls -lh /content/ComfyUI/models/checkpoints/
print('\n✅ Tải model xong! Chạy tiếp Cell 3.')

In [ ]:
# ===== CELL 3: Khởi chạy ComfyUI (chạy NỀN) + tạo link truy cập =====
# Cell này sẽ CHẠY XONG và trả lại quyền điều khiển — ComfyUI vẫn chạy nền.
import subprocess, time, socket, re, os, signal

# Dọn tiến trình cũ nếu chạy lại cell này
!pkill -f "python main.py" 2>/dev/null || true
!pkill -f cloudflared 2>/dev/null || true
time.sleep(2)

# Cài cloudflared (nếu chưa có)
if not os.path.exists('/usr/local/bin/cloudflared'):
    !wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# 1) Chạy ComfyUI NỀN, log ghi ra file
os.chdir('/content/ComfyUI')
comfy_log = open('/content/comfyui.log', 'w')
comfy = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--enable-cors-header'],
    stdout=comfy_log, stderr=subprocess.STDOUT)
print('⏳ Đang khởi động ComfyUI (30-60 giây)...')

# 2) Đợi cổng 8188 mở
for _ in range(180):
    time.sleep(1)
    if comfy.poll() is not None:
        raise RuntimeError('❌ ComfyUI bị tắt! Chạy cell xem log để biết lỗi: !tail -30 /content/comfyui.log')
    try:
        with socket.create_connection(('127.0.0.1', 8188), timeout=1):
            break
    except OSError:
        pass
else:
    raise RuntimeError('❌ Quá 3 phút chưa mở cổng. Xem log: !tail -30 /content/comfyui.log')
print('✅ ComfyUI đã chạy!')

# 3) Chạy cloudflared NỀN (giao thức http2 - hợp mạng VN)
cf_log = open('/content/cloudflared.log', 'w')
cf = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188',
     '--http-host-header', '127.0.0.1:8188', '--protocol', 'http2'],
    stdout=cf_log, stderr=subprocess.STDOUT)

# 4) Đọc link từ log
url = None
for _ in range(60):
    time.sleep(1)
    txt = open('/content/cloudflared.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0)
        break
print()
print('='*60)
if url:
    print('🎨 COPY LINK NÀY, DÁN VÀO THANH ĐỊA CHỈ TAB MỚI:')
    print(url)
else:
    print('⚠️ Chưa lấy được link cloudflare — dùng Cell 5 (localtunnel)')
print('='*60)
print('\n💡 Cell này đã xong nhưng ComfyUI vẫn chạy nền.')
print('   Giờ bạn có thể chạy Cell 4 (kiểm tra) hoặc Cell 5 (link dự phòng).')


In [ ]:
# ===== CELL 4: KIỂM TRA sức khỏe hệ thống (chạy bất cứ lúc nào) =====
# A) ComfyUI còn sống không?
!curl -s -o /dev/null -w "A) ComfyUI noi bo:  HTTP %{http_code} (200 = OK)\n" --max-time 20 http://127.0.0.1:8188/system_stats

# B) Link cloudflare hoạt động không? (tự lấy link từ log)
import re
txt = open('/content/cloudflared.log').read()
m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
if m:
    url = m.group(0)
    print('   Link hien tai:', url)
    !curl -s -o /dev/null -w "B) Qua tunnel:      HTTP %{http_code} (200 = OK, 502 = ComfyUI chet, 000 = tunnel chet)\n" --max-time 40 {url}/system_stats
else:
    print('B) Khong tim thay link trong log cloudflared')

# C) 30 dòng log cuối của ComfyUI
print('\n----- LOG ComfyUI (30 dòng cuối) -----')
!tail -30 /content/comfyui.log


In [ ]:
# ===== CELL 5 (DỰ PHÒNG): Link thay thế qua localtunnel =====
# Dùng khi link trycloudflare bị treo/chặn. ComfyUI phải đang chạy nền (Cell 3 đã xong).
!npm install -g localtunnel > /dev/null 2>&1

import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip()
print('='*60)
print('🔑 MẬT KHẨU (Tunnel Password) khi trang hỏi:', ip)
print('='*60)
print('Đợi link https://....loca.lt hiện ra bên dưới rồi mở link đó.\n')

!lt --port 8188


## 📝 Cách tạo ảnh đầu tiên

1. Mở link `trycloudflare.com` ở trên → giao diện ComfyUI hiện ra với workflow mặc định
2. Ô **Load Checkpoint**: chọn `v1-5-pruned-emaonly-fp16.safetensors`
3. Ô **CLIP Text Encode (Prompt)** phía trên: gõ mô tả ảnh bằng tiếng Anh, ví dụ:
   `a beautiful landscape in Vietnam, rice terraces, sunrise, highly detailed`
4. Ô prompt phía dưới (negative): gõ `blurry, low quality, ugly`
5. Bấm nút **Queue** → đợi vài giây → ảnh hiện ra ở ô **Save Image**

## ❓ Xử lý sự cố
- **Không thấy GPU ở Cell 1** → Runtime → Change runtime type → T4 GPU
- **Link tunnel không hiện** → chạy lại Cell 3
- **Hết phiên/mất kết nối** → Colab free giới hạn ~vài giờ GPU/ngày, chạy lại từ Cell 1